In [2]:
import cv2
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub

# Load MoveNet MultiPose from TF Hub
model = hub.load("https://tfhub.dev/google/movenet/multipose/lightning/1")
movenet = model.signatures['serving_default']


2025-10-01 12:14:56.647683: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-01 12:14:57.278292: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-01 12:14:58.810166: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-01 12:15:04.430764: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [3]:
# Helper: run inference
def detect_poses(frame):
    input_image = tf.image.resize(tf.expand_dims(frame, axis=0), (256, 256))
    input_image = tf.cast(input_image, dtype=tf.int32)

    outputs = movenet(input_image)
    keypoints_with_scores = outputs['output_0'].numpy()  # shape: [1,6,56]
    return keypoints_with_scores[0]  # 6 people max

# # Helper: draw poses
# def draw_poses(frame, keypoints_with_scores, threshold=0.3):
#     h, w, _ = frame.shape
#     for person in keypoints_with_scores:
#         scores = person[2::3]   # confidence scores
#         if np.sum(scores > threshold) < 5:  # skip low-confidence detections
#             continue

#         for i in range(17):
#             y = int(person[i*3+0] * h)
#             x = int(person[i*3+1] * w)
#             conf = person[i*3+2]
#             if conf > threshold:
#                 green = 255 * (i+1)/17
#                 blue = 255 * (1-(i+1)/17)
#                 cv2.circle(frame, (x, y), 4, (0, green, blue), -1)

#     return frame


def draw_poses(frame, keypoints_with_scores, prev_people, fps, threshold=0.4, visualize=True):
    """
    Args:
        frame: current frame (BGR)
        keypoints_with_scores: MoveNet detections [6,56]
        prev_people: list of np.arrays of keypoints from previous frame
        fps: frames per second (for velocity calc)
        threshold: min confidence for joints
        visualize: if True -> draw on frame, if False -> just return data
    Returns:
        frame (possibly annotated),
        new_people (list of keypoints arrays),
        velocities_out (list of (17,2) vx,vy arrays per person)
    """
    h, w, _ = frame.shape
    new_people = []
    velocities_out = []

    for person in keypoints_with_scores:
        scores = person[2::3]
        if np.sum(scores > threshold) < 5:
            continue

        # Reshape to (17,3) => (y, x, conf)
        keypoints = np.array(person[:51]).reshape((17, 3))

        # Scale back to original frame
        keypoints[:, 0] *= h  # y
        keypoints[:, 1] *= w  # x

        new_people.append(keypoints)
        person_velocities = np.zeros((17, 2))

        # Draw keypoints
        if visualize:
            for (y, x, conf) in keypoints:
                if conf > threshold:
                    cv2.circle(frame, (int(x), int(y)), 4, (0, 255, 0), -1)

        # If we have a previous frame, compute velocities
        if prev_people:
            prev_keypoints = min(
                prev_people, key=lambda pk: np.linalg.norm(pk[:, :2] - keypoints[:, :2])
            )
            dt = 1.0 / fps

            for j, ((y, x, conf), (py, px, pconf)) in enumerate(zip(keypoints, prev_keypoints)):
                if conf > threshold and pconf > threshold:
                    vx, vy = (x - px) / dt, (y - py) / dt
                    person_velocities[j] = [vx, vy]

                    if visualize:
                        speed = np.sqrt(vx**2 + vy**2)
                        end_x = int(x + vx * 0.05)
                        end_y = int(y + vy * 0.05)

                        cv2.arrowedLine(
                            frame, (int(x), int(y)), (end_x, end_y),
                            (0, 0, 255), 2, tipLength=0.4
                        )
                        cv2.putText(
                            frame, f"{speed:.1f}", (int(x)+5, int(y)-5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1
                        )

        velocities_out.append(person_velocities)

    return frame, new_people, velocities_out


In [ ]:
# VIDEO_PATH = "data/violent/cam1/1.mp4"
# cap = cv2.VideoCapture(VIDEO_PATH)
# fps = cap.get(cv2.CAP_PROP_FPS)
# w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
# h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# # --- Setup VideoWriter ---
# fourcc = cv2.VideoWriter_fourcc(*"mp4v")  # codec for mp4
# out = cv2.VideoWriter("output_debug.mp4", fourcc, fps, (w, h))

# prev_people = []

# while cap.isOpened():
#     ret, frame = cap.read()
#     if not ret:
#         break

#     rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#     keypoints_with_scores = detect_poses(rgb_frame)

#     # annotated, prev_people = draw_poses(frame.copy(), keypoints_with_scores, prev_people, fps)
#     annotated, prev_people, velocities = draw_poses(frame.copy(), keypoints_with_scores, prev_people, fps, visualize=True)
#     # Show live
#     cv2.imshow("MoveNet MultiPose + Velocities", annotated)

#     # Save annotated frame
#     out.write(annotated)

#     key = cv2.waitKey(30) & 0xFF
#     if key == ord("q"):
#         break

# cap.release()
# out.release()
# cv2.destroyAllWindows()


In [ ]:
VIDEO_PATH = "data/violent/cam1/1.mp4"

# VIDEO_PATH = "me1.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

prev_people = []
all_velocities = []  # store per-frame velocities

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    keypoints_with_scores = detect_poses(rgb_frame)

    # Run without visualization, collect velocities
    _, prev_people, velocities = draw_poses(
        frame, keypoints_with_scores, prev_people, fps, visualize=False
    )
    all_velocities.append(velocities)

    # (Optional) progress display
    current_idx = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
    print(f"Processed frame {current_idx}/{frame_count}", end="\r")

cap.release()

# Save collected data
np.save("velocities.npy", np.array(all_velocities, dtype=object))
print("\nSaved velocities to velocities.npy")


Processed frame 145/145
Saved velocities to velocities.npy


In [8]:
velocities = np.load("velocities.npy", allow_pickle=True)
print(velocities.shape)       # (num_frames,)
# print(len(velocities[10]))     # number of people in first frame
# print(velocities[0][0].shape) # (17, 2) for first person


(145,)


In [23]:
velocities[1][2]

array([[   0.        ,    0.        ],
       [   0.        ,    0.        ],
       [   0.        ,    0.        ],
       [  28.83544731,   16.22772217],
       [   0.        ,    0.        ],
       [-133.15795898,  -15.9695425 ],
       [   0.        ,    0.        ],
       [   0.        ,    0.        ],
       [   0.        ,    0.        ],
       [   0.        ,    0.        ],
       [   0.        ,    0.        ],
       [   0.        ,    0.        ],
       [ 282.49145508, -141.73278809],
       [   0.        ,    0.        ],
       [   0.        ,    0.        ],
       [   0.        ,    0.        ],
       [   0.        ,    0.        ]])